<img align="left" src = https://project.lsst.org/sites/default/files/Rubin-O-Logo_0.png width=250 style="padding: 10px"> 
<br>
<b>Notebook for testing a tool to retrieve template epoch history for diaSources</b> <br>
Contact author: Ian Sullivan<br>
Last verified to run: 1 October 2026<br>
LSST Science Piplines version: d_2026_09_08<br>
Run with a local ap_verify run of ap_verify_ci_hits2015

### 1. Main package imports

In [1]:
import os
import importlib
import pprint

In [2]:
import matplotlib.pyplot as plt
#%matplotlib widget
%matplotlib inline

import numpy as np
import pandas as pd

In [3]:
import astropy.units as u
from astropy.coordinates import SkyCoord

In [4]:
import lsst.afw.display as afwDisplay
import lsst.geom

import lsst.daf.butler as dafButler
import lsst.pipe.base
from lsst.pipe.base import Instrument

In [5]:
!eups list analysis_ap

   LOCAL:/Users/sullivan/LSST/code/build/analysis_ap 	setup
   g5076a79c9e+0c6e8084b9 	w_latest w_2026_36
   g5076a79c9e+7042551b87 	d_2026_08_17
   g5076a79c9e+ecf2d831d5 	d_2026_08_29 d_2026_08_30
   gcedecddff9+9d3c2caaf6 	d_2026_09_08
   gcedecddff9+aa32c78b00 	d_latest current d_2026_09_16


In [6]:
from lsst.analysis.ap import apdb
from lsst.analysis.ap import extract_timestamped_messages

        Use pytest instead. [astropy.tests.runner]
        Use pytest instead. [astropy.utils.decorators]


In [7]:
from lsst.analysis.ap import nb_utils
from lsst.analysis.ap import display_images, cutout_grid

In [8]:
from lsst.analysis.ap import apdb, extract_timestamped_messages, plotDiaSourceLightcurve
from lsst.analysis.ap import nb_utils, subtraction_quality_report
from lsst.analysis.ap import display_images, display_images_ab, lightcurve, cutout_grid, summarize_run, collect_task_runtimes, display_coadd_coverage

In [10]:
from lsst.analysis.ap import get_template_inputs, summarize_template_inputs

In [11]:
from astropy.table import Table

In [12]:
importlib.reload(lsst.analysis.ap)

<module 'lsst.analysis.ap' from '/Users/sullivan/LSST/code/build/analysis_ap/python/lsst/analysis/ap/__init__.py'>

In [13]:
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')


#### Define the repo for the test data

In [14]:
!pwd

/Users/sullivan/LSST/code/ap_pipe-notebooks/notebooks


In [15]:
path = '/Users/sullivan/LSST/code/ap_verify_output/d_2026_09_16/DECam/DM-56200-future-10'
repo = path + '/repo/'
collections = 'ap_verify-output'
instrument = 'DECam'
skymap = 'decam_rings_v1'
# skymap = 'lsst_cells_v2'
# skymap = 'hsc_rings_v1'

butler = dafButler.Butler(repo, collections=collections)

dbType='sqlite'
dbName = path + "/association.db"

reader = apdb.ApdbSqliteQuery(dbName, instrument=instrument)

In [16]:
len(reader._tables)

5

# Verify the code that was actually used

In [17]:
pkgs = butler.get("packages")
print(pkgs["ip_diffim"])    

LOCAL:/Users/sullivan/LSST/code/build/ip_diffim@2ae20adc63bb1a814fcd35b82993e0b5d647b275+03afffcabfe00ba8e9b12f73da45a090


In [18]:
visit_list = [411420, 419802, 411420, 419802, 411371, 411371]
detector_list = [5, 5, 10, 10, 56, 60]

In [19]:
visit = visit_list[0]
detector = detector_list[0]

In [20]:
science = butler.get("preliminary_visit_image", visit=visit, detector=detector)

In [23]:
template = butler.get("template_detector", visit=visit, detector=detector)

In [25]:
diaSources = butler.get("dia_source_detector", visit=visit, detector=detector)

In [27]:
inputs = get_template_inputs(butler, diaSources)
summary = summarize_template_inputs(inputs)


In [29]:
inputs

diaSourceId,visit,detector,tract,patch,input_visit,input_detector,coadd_weight,mid_mjd_tai,exposure_time,weight
int64,int64,int64,int64,int64,int64,int64,float64,float64,float64,float64
1380496609116164,411420,5,8604,2,288976,5,20.632702078407647,56717.1478373734,174.0,0.28067971504979133
1380496609116164,411420,5,8604,2,289016,5,20.927586861956733,56717.24317227713,174.0,0.2846912195393412
1380496609116164,411420,5,8604,2,289409,5,16.469007671939462,56719.12226149996,160.0,0.22403834276996176
1380496609116164,411420,5,8604,2,289871,5,15.480476171766076,56721.22766281642,160.0,0.21059072264090586
1380496609116165,411420,5,8604,8,288976,5,19.16649923128215,56717.1478373734,174.0,0.23368968122497358
1380496609116165,411420,5,8604,8,289016,5,18.88527660413566,56717.24317227713,174.0,0.23026084295367097
1380496609116165,411420,5,8604,8,289409,5,15.36386051392569,56719.12226149996,160.0,0.18732558421645965
1380496609116165,411420,5,8604,8,289820,5,14.38295715012641,56721.114138368095,160.0,0.17536581046577748
1380496609116165,411420,5,8604,8,289871,5,14.218289448949685,56721.22766281642,160.0,0.1733580811391182


In [28]:
summary

diaSourceId,n_inputs,mean_mjd_tai,min_mjd_tai,max_mjd_tai
int64,int64,float64,float64,float64
1380496609116164,4,56718.476498480944,56717.1478373734,56721.22766281642
1380496609116165,5,56718.94247372036,56717.1478373734,56721.22766281642
1380496609116166,5,56718.708904518084,56717.1478373734,56721.22766281642
1380496609116173,5,56718.71404336415,56717.1478373734,56721.22766281642
1380496609116195,5,56718.94247372036,56717.1478373734,56721.22766281642
1380496609116205,5,56718.94247372036,56717.1478373734,56721.22766281642
1380496609116210,11,56718.807754684494,56717.14547216891,56721.22766281642
1380496609116211,5,56718.72201800679,56717.1478373734,56721.22766281642
1380496609116227,11,56718.807754684494,56717.14547216891,56721.22766281642
